# 01 Analytical Base Table (ABT)

**Project:** NYC Yellow Taxi Fleet Deployment and Revenue Optimization (DSS142P, Group 1)

**Input:** `data/NYC_Taxi_Cleaned_Analysis_Ready.csv`: TLC yellow taxi trips, Jan–Nov 2025, pre-cleaned (durations 1–180 min, speeds 1–80 mph).
**Output:** `data/NYC_Taxi_ABT.parquet` (+ `.csv`): one row per valid trip, with the revenue, fee and time fields used by `02_fleet_optimization.ipynb`.
**Run order:** this notebook → `02_fleet_optimization.ipynb`

| Section | Purpose |
|---|---|
| 1 | Setup |
| 2 | Load the input |
| 3 | Validate the input (no changes yet) |
| 4 | Row filters |
| 5 | Fee corrections |
| 6 | Engineered fields |
| 7 | Final checks |
| 8 | Save |

# 1. Setup

In [74]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

notebook_start = time.perf_counter()

# ---- Paths (relative to this notebook's folder) ----
DATA_DIR = Path("../data")
INPUT_CSV = DATA_DIR / "NYC_Taxi_Cleaned_Analysis_Ready.csv"
OUTPUT_PARQUET = DATA_DIR / "NYC_Taxi_ABT.parquet"
OUTPUT_CSV = DATA_DIR / "NYC_Taxi_ABT.csv"

# ---- Study constants ----
STUDY_YEAR = 2025  # CBD congestion pricing started Jan 5, 2025
FEE_COLS = ["cbd_congestion_fee", "congestion_surcharge", "Airport_fee",
            "mta_tax", "improvement_surcharge"]  # the 5 fees deducted from gross revenue

pd.set_option("display.max_columns", 50)

print("Input found:", INPUT_CSV.exists())
print("Input path: ", INPUT_CSV.resolve())
print(f"pandas {pd.__version__} | numpy {np.__version__}")

Input found: True
Input path:  /Users/alfonsi/Downloads/nyc-taxi-original/data/NYC_Taxi_Cleaned_Analysis_Ready.csv
pandas 3.0.6 | numpy 2.5.3


## 2. Load the input

In [75]:
dtypes = {
    "VendorID": "int8", "passenger_count": "int8", "RatecodeID": "int16",
    "store_and_fwd_flag": "category", "PULocationID": "int16", "DOLocationID": "int16",
    "payment_type": "int8", "pickup_hour": "int8", "day_of_week": "category",
}

df = pd.read_csv(
    INPUT_CSV,
    dtype=dtypes,
    parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"],
)
n_input = len(df)

print(f"Rows: {n_input:,} | Columns: {df.shape[1]}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
df.head()

Rows: 3,946,698 | Columns: 25
Memory: 0.49 GB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duration_min,speed_mph,pickup_hour,day_of_week,is_weekend
0,2,2025-01-18 20:53:30,2025-01-18 21:00:47,1,0.97,1,N,238,166,1,8.6,1.00,0.5,2.22,0.0,1.0,13.32,0.0,0.0,0.00,7.283333,7.990847,20,Saturday,True
1,1,2025-01-25 11:12:51,2025-01-25 11:17:57,1,0.60,1,N,50,48,2,5.8,3.25,0.5,0.00,0.0,1.0,10.55,2.5,0.0,0.75,5.100000,7.058824,11,Saturday,True
2,1,2025-01-21 15:09:31,2025-01-21 15:19:02,1,0.80,1,N,236,237,1,9.3,2.50,0.5,3.35,0.0,1.0,16.65,2.5,0.0,0.00,9.516667,5.043783,15,Tuesday,False
3,2,2025-01-11 22:25:45,2025-01-11 22:34:22,2,1.93,1,N,231,68,1,10.7,1.00,0.5,3.92,0.0,1.0,20.37,2.5,0.0,0.75,8.616667,13.439072,22,Saturday,True
4,2,2025-01-04 23:37:07,2025-01-04 23:45:58,1,4.44,1,N,137,88,0,26.3,0.00,0.5,4.82,0.0,1.0,35.12,0.0,NaN,0.00,8.850000,30.101695,23,Saturday,True


## 3. Validate the input
Nothing is changed in this section. Each check documents the state of the data so that every fix in Sections 4–5 can be traced back to a finding here.

In [76]:
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
structure

,dtype,missing,missing_%,n_unique
VendorID,int8,0,0.00,3
tpep_pickup_datetime,datetime64[us],0,0.00,3614373
tpep_dropoff_datetime,datetime64[us],0,0.00,3612982
passenger_count,int8,0,0.00,10
trip_distance,float64,0,0.00,3521
RatecodeID,int16,0,0.00,7
store_and_fwd_flag,category,0,0.00,2
PULocationID,int16,0,0.00,258
DOLocationID,int16,0,0.00,257
payment_type,int8,0,0.00,5


In [77]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Pickup range:", df["tpep_pickup_datetime"].min(), "→", df["tpep_pickup_datetime"].max())
print("Dropoff not after pickup:", (df["tpep_dropoff_datetime"] <= df["tpep_pickup_datetime"]).sum())

print("\nTrips per month:")
print(df["tpep_pickup_datetime"].dt.to_period("M").value_counts().sort_index())

Exact duplicate rows: 0
Pickup range: 2007-12-05 18:45:00 → 2025-11-30 23:59:59
Dropoff not after pickup: 0

Trips per month:
tpep_pickup_datetime
2007-12         1
2009-01         1
2024-12         1
2025-01    320328
2025-02    325784
2025-03    376925
2025-04    361820
2025-05    402670
2025-06    379991
2025-07    343031
2025-08    312404
2025-09    377181
2025-10    387774
2025-11    358787
Freq: M, Name: count, dtype: int64


In [78]:
df.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).T.round(2)

,count,mean,min,0.1%,1%,50%,99%,99.9%,max,std
VendorID,3946698.0,1.787345,1.0,1.0,1.0,2.0,2.0,2.0,6.0,0.419435
tpep_pickup_datetime,3946698,2025-06-18 17:28:59.252737,2007-12-05 18:45:00,2025-01-01 13:03:07.576000,2025-01-05 18:14:37.550000,2025-06-15 11:17:25,2025-11-26 15:35:20.030000,2025-11-30 16:15:12.909000,2025-11-30 23:59:59,NaN
tpep_dropoff_datetime,3946698,2025-06-18 17:45:57.706723,2007-12-05 19:02:00,2025-01-01 13:15:50.788000,2025-01-05 18:29:55.760000,2025-06-15 11:31:06,2025-11-26 15:58:33.450000,2025-11-30 16:35:21.303000,2025-12-01 00:29:37,NaN
passenger_count,3946698.0,1.233373,0.0,0.0,1.0,1.0,4.0,6.0,9.0,0.659559
trip_distance,3946698.0,3.441244,0.11,0.18,0.33,1.9,19.27,26.88,99.06,4.121952
RatecodeID,3946698.0,2.710091,1.0,1.0,1.0,1.0,99.0,99.0,99.0,12.641342
PULocationID,3946698.0,162.071813,1.0,4.0,13.0,161.0,263.0,263.0,263.0,65.624391
DOLocationID,3946698.0,161.15364,1.0,1.0,10.0,162.0,263.0,263.0,263.0,70.004204
payment_type,3946698.0,0.946666,0.0,0.0,0.0,1.0,4.0,4.0,4.0,0.635978
fare_amount,3946698.0,19.368593,0.01,0.4,4.4,14.2,70.0,103.1,1334.02,15.765093


In [79]:
for col in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index().to_string())


VendorID:
VendorID
1     845989
2    3099033
6       1676

RatecodeID:
RatecodeID
1     3743094
2      108550
3       10124
4        1046
5       17056
6           1
99      66827

payment_type:
payment_type
0     757387
1    2747886
2     382115
3      13151
4      46159

store_and_fwd_flag:
store_and_fwd_flag
N    3937169
Y       9529


In [80]:
components = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
              "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]

residual = (df["total_amount"] - df[components].fillna(0).sum(axis=1)).round(2)

(df.assign(residual=residual, adds_up=residual.abs() < 0.011)
   .groupby(["VendorID", "payment_type"])
   .agg(trips=("residual", "size"),
        pct_adds_up=("adds_up", "mean"),
        typical_gap=("residual", "median"))
   .assign(pct_adds_up=lambda t: (t["pct_adds_up"] * 100).round(1)))

trips  pct_adds_up  typical_gap
VendorID payment_type                                   
1        0              139896         19.4         2.50
         1              613816         12.2        -3.25
         2               81632          4.7        -3.25
         3                6338          5.6        -3.25
         4                4307          6.3        -3.25
2        0              615815         20.6         2.50
         1             2134070         99.9         0.00
         2              300483         99.7         0.00
         3                6813         99.9         0.00
         4               41852         99.8         0.00
6        0                1676          0.0        25.72

### 3.1 Validation findings

| ID | Finding | Evidence (cells above) | Action |
|---|---|---|---|
| F1 | 3 trips fall outside 2025 (2007-12, 2009-01, 2024-12) | Trips per month | Drop: meter clock errors (§4) |
| F2 | ~320–400k trips per month, about a tenth of TLC's published monthly volume | Trips per month | **Confirm with group:** was the data sampled upstream? Document only |
| F3 | `Airport_fee` is missing for 757,387 trips (19.2%), and every one is a flex-fare trip (`payment_type` = 0) | Structure + code counts | Fill with 0 and correct the FA2 note that these were non-airport street hails (§5) |
| F4 | Flex-fare trips record `congestion_surcharge` = 0, but `total_amount` is typically $2.50 higher than its parts | Total check: gap +2.50 | Restore $2.50 where the gap is exactly 2.50 (§5) |
| F5 | Vendor 1 totals are typically $3.25 *less* than their parts: `extra` already contains the congestion surcharge and CBD fee | Total check: gap −3.25 | `total_amount` is correct, so net revenue is unaffected. Never use `extra` as a feature (§5 note) |
| F6 | Vendor 6: 1,676 trips, all flex fare, totals ~$26 off, 0% add up | Total check | Drop: unreliable records (§4) |
| F7 | `RatecodeID` = 99 (unknown) on 66,827 trips | Code counts | Drop: outside TLC codes 1–6 (§4) |
| F8 | `passenger_count` = 0 on some trips (minimum is 0) | Numeric ranges | Drop: invalid trip record (§4) |
| F9 | `payment_type` 3 (no charge) = 13,151 and 4 (dispute) = 46,159 | Code counts | Drop: not realized revenue (§4) |
| F10 | Negative values in `extra` (min −17.39), `mta_tax` (min −7.81) and `cbd_congestion_fee` (min −0.75) | Numeric ranges | Drop these refund/correction rows instead of clipping (§4) |
| F11 | No duplicates, no dropoff before pickup, durations 1–180 min, speeds 1–80 mph | Duplicates/dates + ranges | No action: confirms the upstream cleaning |
| F12 | 22,852 standard-rate trips have a fare below the $3.00 initial meter charge (median $1.28 over ~3.3 mi); all but one are flex-fare | Found in §7 review | Drop: implausible fare records (§4) |
| F13 | RatecodeID 5 (negotiated, 15,924 trips) produces most extreme $/hr values (19% exceed $300/hr; max $14,722/hr); RatecodeID 6 has 1 trip | Found in §7 review | Drop: prearranged prices are not a deployment target (§4) |
| F14 | Tips are recorded on 95% of card trips but only 8% of flex-fare and 0% of cash trips | Found in §7 review | Keep tips (D2); add `net_revenue_ex_tip` for robustness checks (§6) |

## 4. Row filters

Each filter removes one type of invalid trip identified in §3.1. Filters run in sequence, so a trip that breaks several rules is counted once, under the first rule it fails. The log at the end accounts for every input row.

In [81]:
filter_log = []

def apply_filter(data, keep_mask, finding, rule):
    """Keep rows where keep_mask is True, and record how many were dropped."""
    n_before = len(data)
    data = data.loc[keep_mask].copy()
    filter_log.append({
        "finding": finding,
        "rule": rule,
        "rows_before": n_before,
        "dropped": n_before - len(data),
        "rows_after": len(data),
    })
    return data

In [82]:
charge_cols = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
               "improvement_surcharge", "congestion_surcharge", "Airport_fee",
               "cbd_congestion_fee", "total_amount"]

df = apply_filter(df, df["tpep_pickup_datetime"].dt.year == STUDY_YEAR,
                  "F1", "Pickup year is 2025")
df = apply_filter(df, df["VendorID"] != 6,
                  "F6", "Exclude Vendor 6 (totals don't reconcile)")
df = apply_filter(df, (df[charge_cols].fillna(0) >= 0).all(axis=1),
                  "F10", "No negative charge components (refunds/corrections)")
df = apply_filter(df, df["RatecodeID"].between(1, 6),
                  "F7", "RatecodeID within TLC codes 1–6")
df = apply_filter(df, df["passenger_count"] > 0,
                  "F8", "At least one passenger")
df = apply_filter(df, ~df["payment_type"].isin([3, 4]),
                  "F9", "Exclude no-charge (3) and disputed (4) trips")
df = apply_filter(df, ~((df["RatecodeID"] == 1) & (df["fare_amount"] < 3.00)),
                  "F12", "Standard-rate fare at least the $3.00 initial charge")
df = apply_filter(df, df["RatecodeID"].isin([1, 2, 3, 4]),
                  "F13", "Exclude negotiated (5) and group-ride (6) fares")

In [83]:
log = pd.DataFrame(filter_log)
log["dropped_%_of_input"] = (log["dropped"] / n_input * 100).round(2)
display(log)

n_dropped = n_input - len(df)
print(f"Input rows:   {n_input:,}")
print(f"Dropped rows: {n_dropped:,} ({n_dropped / n_input:.2%})")
print(f"Kept rows:    {len(df):,} ({len(df) / n_input:.2%})")

assert log["dropped"].sum() == n_dropped, "Row accounting does not balance!"
print("Row accounting balances ✓")

,finding,rule,rows_before,dropped,rows_after,dropped_%_of_input
0,F1,Pickup year is 2025,3946698,3,3946695,0.00
1,F6,Exclude Vendor 6 (totals don't reconcile),3946695,1676,3945019,0.04
2,F10,No negative charge components (refunds/correct...,3945019,75,3944944,0.00
3,F7,RatecodeID within TLC codes 1–6,3944944,66827,3878117,1.69
4,F8,At least one passenger,3878117,22572,3855545,0.57
5,F9,Exclude no-charge (3) and disputed (4) trips,3855545,58873,3796672,1.49
6,F12,Standard-rate fare at least the $3.00 initial ...,3796672,22852,3773820,0.58
7,F13,Exclude negotiated (5) and group-ride (6) fares,3773820,15925,3757895,0.40


Input rows:   3,946,698
Dropped rows: 188,803 (4.78%)
Kept rows:    3,757,895 (95.22%)
Row accounting balances ✓


## 5. Fee corrections

Two recording problems from §3.1 are corrected here. Rows are not removed; values are fixed so that the fee columns match what the passenger was actually charged.

In [84]:
parts_sum = df[components].fillna(0).sum(axis=1)
gap = (df["total_amount"] - parts_sum).round(2)

restore = (
    (df["payment_type"] == 0)            # flex-fare trip
    & (df["congestion_surcharge"] == 0)  # surcharge recorded as 0
    & (gap == 2.50)                      # but the total is exactly $2.50 higher
)
df.loc[restore, "congestion_surcharge"] = 2.50

print(f"F4: congestion surcharge restored on {restore.sum():,} flex-fare trips")

F4: congestion surcharge restored on 577,980 flex-fare trips


In [85]:
missing_airport = df["Airport_fee"].isna()
print(f"F3: Airport_fee blanks: {missing_airport.sum():,}")
print(f"    Share of blanks that are flex-fare trips: {df.loc[missing_airport, 'payment_type'].eq(0).mean():.1%}")

df["Airport_fee"] = df["Airport_fee"].fillna(0)
print(f"    Blanks remaining after fill: {df['Airport_fee'].isna().sum()}")

F3: Airport_fee blanks: 732,786
    Share of blanks that are flex-fare trips: 100.0%
    Blanks remaining after fill: 0


In [86]:
v1_card_cash = (df["VendorID"] == 1) & (df["payment_type"] != 0)

expected_total = np.where(
    v1_card_cash,
    # F5: Vendor 1's 'extra' already contains the congestion, CBD and airport fees
    df[["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge"]].sum(axis=1),
    df[components].sum(axis=1),
)
reconciles = (df["total_amount"] - expected_total).round(2).abs() < 0.011

display(
    df.assign(adds_up=reconciles)
      .groupby(["VendorID", "payment_type"])["adds_up"]
      .agg(trips="size", pct_adds_up="mean")
      .assign(pct_adds_up=lambda t: (t["pct_adds_up"] * 100).round(1))
)
print(f"Overall: {reconciles.mean():.2%} of trips reconcile | {(~reconciles).sum():,} do not")

trips  pct_adds_up
VendorID payment_type                      
1        0              139824         99.9
         1              527171        100.0
         2               78283        100.0
2        0              592962         99.0
         1             2120602         99.9
         2              299053         99.7

Overall: 99.74% of trips reconcile | 9,875 do not


### 5.1 Fee correction notes

- **F4:** $2.50 congestion surcharge restored on flex-fare trips whose total was exactly $2.50 above the sum of their components.
- **F3:** `Airport_fee` blanks occur only on flex-fare trips; filled with 0. (FA2 described these as non-airport street hails, which was incorrect.)
- **F5:** For Vendor 1 card/cash trips, `extra` already includes the congestion surcharge, CBD fee and airport fee. `total_amount` is correct, so net revenue is unaffected, but `extra` must not be used as a stand-alone variable.
- **Remaining:** about 0.4% of trips still don't reconcile exactly (mostly Vendor 2 flex-fare trips with irregular gaps). They are kept, because `total_amount` is what the passenger actually paid.

## 6. Engineered fields

| ID | Decision | Reason |
|---|---|---|
| D1 | `net_revenue` = gross − 5 fees − tolls | Tolls are pass-through (bridge/tunnel charges), not driver income |
| D2 | Tips stay in `net_revenue` | Real driver income; cash tips are unrecorded (stated limitation) |
| D3 | Fare tiers use fixed cutoffs on `fare_amount` | `total_amount` includes the fees AQ2 measures; fixed cutoffs can become a minimum-fare rule |
| D4 | Trip-length tiers use fixed mile cutoffs | Interpretable thresholds for AQ2 |
| D5 | No fixed congestion bins; keep `speed_mph` | AQ3 must find the congestion threshold from the data, not assume it |

`net_revenue_per_hour` is **not capped** here. Outlier handling is a modeling decision and is made in `02_fleet_optimization.ipynb`.

In [87]:
df["gross_revenue"] = df["total_amount"]
df["total_fees"] = df[FEE_COLS].sum(axis=1)
df["congestion_fees"] = df["cbd_congestion_fee"] + df["congestion_surcharge"]

df["net_revenue"] = df["gross_revenue"] - df["total_fees"] - df["tolls_amount"]   # D1
df["net_revenue_ex_tip"] = df["net_revenue"] - df["tip_amount"]                   # F14 robustness
df["net_revenue_per_hour"] = df["net_revenue"] / df["duration_min"] * 60          # uncapped

df["fee_impact_pct"] = df["total_fees"] / df["gross_revenue"] * 100
df["congestion_fee_pct"] = df["congestion_fees"] / df["gross_revenue"] * 100

# Self-checks: every value must be valid before anything downstream uses it
assert (df["gross_revenue"] > 0).all(), "Zero or negative gross revenue"
assert (df["duration_min"] > 0).all(), "Zero duration would divide by zero"
assert (df["net_revenue"] > 0).all(), "Net revenue must be positive"
assert df[["total_fees", "net_revenue", "net_revenue_per_hour", "fee_impact_pct"]].notna().all().all()

df[["gross_revenue", "total_fees", "congestion_fees", "tolls_amount",
    "net_revenue", "net_revenue_ex_tip", "net_revenue_per_hour",
    "fee_impact_pct", "congestion_fee_pct"]].describe().round(2)

,gross_revenue,total_fees,congestion_fees,tolls_amount,net_revenue,net_revenue_ex_tip,net_revenue_per_hour,fee_impact_pct,congestion_fee_pct
count,3757895.00,3757895.00,3757895.00,3757895.00,3757895.00,3757895.00,3757895.00,3757895.00,3757895.00
mean,27.88,4.47,2.85,0.48,22.93,19.82,91.48,20.68,13.40
std,19.42,0.88,0.91,1.94,17.96,15.49,41.49,9.16,6.92
min,4.50,0.00,0.00,0.00,3.00,3.00,2.05,0.00,0.00
25%,16.65,4.75,2.50,0.00,12.11,10.30,70.63,14.24,9.41
50%,21.75,4.75,3.25,0.00,17.19,14.90,84.06,20.52,13.78
75%,30.47,4.75,3.25,0.00,25.91,22.90,103.09,26.84,17.99
max,560.40,14.75,4.00,115.87,558.90,411.80,5889.84,63.73,41.94


In [88]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.normalize()   # date with time set to 00:00
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month.astype("int8")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_of_week"] = pd.Categorical(df["day_of_week"].astype(str), categories=day_order, ordered=True)

# Self-checks: the time fields from the earlier cleaning step must match the timestamp
ts = df["tpep_pickup_datetime"]
assert (df["pickup_hour"] == ts.dt.hour).all(), "pickup_hour mismatch"
assert (df["day_of_week"].astype(str) == ts.dt.day_name()).all(), "day_of_week mismatch"
assert (df["is_weekend"] == (ts.dt.dayofweek >= 5)).all(), "is_weekend mismatch"
print("Time fields consistent with the pickup timestamp ✓")
print(df["pickup_month"].value_counts().sort_index().to_string())

Time fields consistent with the pickup timestamp ✓
pickup_month
1     308589
2     315088
3     364210
4     348376
5     384718
6     365964
7     328019
8     292101
9     358195
10    362391
11    330244


In [89]:
FARE_BINS = [0, 10, 20, 40, np.inf]
FARE_LABELS = ["Under $10", "$10–20", "$20–40", "$40+"]

DIST_BINS = [0, 1, 3, 7, np.inf]
DIST_LABELS = ["Under 1 mi", "1–3 mi", "3–7 mi", "7+ mi"]

df["fare_tier"] = pd.cut(df["fare_amount"], bins=FARE_BINS, labels=FARE_LABELS, right=False)          # D3
df["trip_length_tier"] = pd.cut(df["trip_distance"], bins=DIST_BINS, labels=DIST_LABELS, right=False)  # D4

assert df[["fare_tier", "trip_length_tier"]].notna().all().all(), "Some trips fell outside the tiers"

In [90]:
for tier in ["fare_tier", "trip_length_tier"]:
    display(
        df.groupby(tier, observed=True)
          .agg(trips=("net_revenue", "size"),
               avg_fees=("total_fees", "mean"),
               fee_pct_of_gross=("fee_impact_pct", "mean"),
               net_per_hour=("net_revenue_per_hour", "mean"))
          .round(2)
    )

,trips,avg_fees,fee_pct_of_gross,net_per_hour
fare_tier,,,,
Under $10,975786,4.36,31.12,110.10
$10–20,1676751,4.49,21.16,80.85
$20–40,778135,4.42,12.61,83.88
$40+,327223,4.78,6.24,108.50


,trips,avg_fees,fee_pct_of_gross,net_per_hour
trip_length_tier,,,,
Under 1 mi,774934,4.42,30.90,113.13
1–3 mi,1851017,4.48,22.09,83.02
3–7 mi,679777,4.36,14.10,80.13
7+ mi,452167,4.64,7.26,106.08


### 6.1 Notes
- Fees are nearly flat per trip (~$4.40), so they take a much larger share of short, low fares than of long ones. This is the starting point for AQ2.
- Short trips show high net $/hr because they are quick; whether they are worth chasing depends on how long a driver waits between them (demand), which is handled in AQ1.
- `extra` is kept as a source column but must not be used as a variable (F5).

In [91]:
checks = {
    "Row count matches the filter log":    len(df) == filter_log[-1]["rows_after"],
    "No missing values in any column":     df.isna().sum().sum() == 0,
    "No duplicate rows":                   not df.duplicated().any(),
    "All pickups within Jan–Nov 2025":     df["tpep_pickup_datetime"].between("2025-01-01", "2025-12-01").all(),
    "Vendors are 1 or 2 only (F6)":        df["VendorID"].isin([1, 2]).all(),
    "Rate codes are 1–4 only (F7, F13)":   df["RatecodeID"].isin([1, 2, 3, 4]).all(),
    "Payment types are 0, 1, 2 only (F9)": df["payment_type"].isin([0, 1, 2]).all(),
    "At least one passenger (F8)":         (df["passenger_count"] > 0).all(),
    "No negative charges (F10)":           (df[charge_cols] >= 0).all().all(),
    "Standard fares at least $3 (F12)":    (df.loc[df["RatecodeID"] == 1, "fare_amount"] >= 3).all(),
    "Net revenue positive (D1)":           (df["net_revenue"] > 0).all(),
    "Fee shares between 0 and 100%":       df[["fee_impact_pct", "congestion_fee_pct"]].apply(lambda s: s.between(0, 100)).all().all(),
}

results = pd.Series(checks, name="passed")
display(results.to_frame())
assert results.all(), f"Failed checks: {list(results[~results].index)}"
print(f"All {len(results)} checks passed ✓ | Final ABT: {len(df):,} rows × {df.shape[1]} columns")

,passed
Row count matches the filter log,True
No missing values in any column,True
No duplicate rows,True
All pickups within Jan–Nov 2025,True
Vendors are 1 or 2 only (F6),True
"Rate codes are 1–4 only (F7, F13)",True
"Payment types are 0, 1, 2 only (F9)",True
At least one passenger (F8),True
No negative charges (F10),True
Standard fares at least $3 (F12),True


All 12 checks passed ✓ | Final ABT: 3,757,895 rows × 37 columns


In [92]:
nrph = df["net_revenue_per_hour"]
print("Net revenue per occupied hour (uncapped):")
print(nrph.describe(percentiles=[0.01, 0.5, 0.99, 0.999]).round(2).to_string())
print(f"\nTrips above $300/hr: {(nrph > 300).sum():,} ({(nrph > 300).mean():.2%})")
print(f"Trips above $500/hr: {(nrph > 500).sum():,} ({(nrph > 500).mean():.2%})")

Net revenue per occupied hour (uncapped):
count    3757895.00
mean          91.48
std           41.49
min            2.05
1%            42.67
50%           84.06
99%          208.24
99.9%        360.73
max         5889.84

Trips above $300/hr: 7,421 (0.20%)
Trips above $500/hr: 1,655 (0.04%)


### 7.1 Handoff notes for `02_fleet_optimization.ipynb`
- **Target outliers:** `net_revenue_per_hour` is uncapped; the outlier rule is decided in notebook 02 (see figures above).
- **Tip bias (F14):** tips are effectively card-only; use `net_revenue_ex_tip` to check that zone rankings are not driven by payment mix.
- **`extra` (F5):** do not use as a variable.
- **Sampling (F2):** pending group confirmation; affects how trip counts are interpreted as demand.

## 7. Final checks
Every rule from §3–§6 is verified on the finished table. Any failure stops the notebook before the ABT is saved.

## 8. Save

The ABT is saved twice:
- **Parquet** (primary): much smaller and faster to load, and it keeps data types (dates, ordered categories), so notebook 02 doesn't need to re-parse anything.
- **CSV** (compatibility): for anyone opening it in Excel or another tool.

In [93]:
df = df.reset_index(drop=True)

df.to_parquet(OUTPUT_PARQUET, index=False)
df.to_csv(OUTPUT_CSV, index=False)

# Verify: read the Parquet file back and compare it with what's in memory
reloaded = pd.read_parquet(OUTPUT_PARQUET)
assert reloaded.shape == df.shape, "Row/column count changed on save"
assert list(reloaded.columns) == list(df.columns), "Column order changed on save"
pd.testing.assert_frame_equal(reloaded.sample(10_000, random_state=42),
                              df.sample(10_000, random_state=42),
                              check_dtype=False)

for path in [OUTPUT_PARQUET, OUTPUT_CSV]:
    print(f"Saved {path.name}: {path.stat().st_size / 1024**2:,.0f} MB")
print(f"Verified: {reloaded.shape[0]:,} rows × {reloaded.shape[1]} columns ✓")

Saved NYC_Taxi_ABT.parquet: 197 MB
Saved NYC_Taxi_ABT.csv: 976 MB
Verified: 3,757,895 rows × 37 columns ✓


In [94]:
elapsed = time.perf_counter() - notebook_start
print(f"Notebook runtime: {int(elapsed // 60)}m {elapsed % 60:.0f}s")
print(f"Input:  {n_input:,} trips")
print(f"Output: {len(df):,} trips × {df.shape[1]} columns ({len(df) / n_input:.2%} kept)")

Notebook runtime: 0m 52s
Input:  3,946,698 trips
Output: 3,757,895 trips × 37 columns (95.22% kept)
